In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}
document.title="AiiDAlab EXAFS Code Setup"

In [ ]:
from aiida import load_profile
load_profile();

# AiiDAlab EXAFS: Resource & Code Setup

Configure local and remote computational resources (including STFC SCARF and local FEFF8L binaries) for EXAFS workflows.

### 1. Localhost FEFF Setup

Automatically register `feff@localhost` using the FEFF8L executable bundled with `xraylarch`, as well as `python3@localhost` for MD path aggregation.

In [ ]:
import ipywidgets as ipw
from aiidalab_exafs.codes import setup_feff_code, setup_python3_code

status_out = ipw.Output()
setup_btn = ipw.Button(
    description="Set up Local FEFF Code",
    button_style="primary",
    icon="wrench",
    layout=ipw.Layout(width="220px", height="36px"),
)

def _on_click(_):
    with status_out:
        status_out.clear_output()
        try:
            feff_code, feff_created = setup_feff_code()
            py_code, py_created = setup_python3_code()
            print(f"✓ FEFF code ready: {feff_code.full_label}")
            print(f"✓ Python code ready: {py_code.full_label}")
        except Exception as exc:
            print(f"✗ Error: {exc}")

setup_btn.on_click(_on_click)
display(ipw.VBox([setup_btn, status_out], layout=ipw.Layout(margin="8px 0 16px 0")))

### 2. Configure Remote HPC Cluster (SCARF / STFC)

Configure remote computer connections and code instances using the shared STFC/UKRI HPC resource database (`remotes.json`).

In [ ]:
from alc_aiidalab_widgets.widgets.code_setup import CodeSetupWidget
display(CodeSetupWidget())

### 3. Configured AiiDA Codes

Review all configured codes currently stored in your AiiDA profile.

In [ ]:
from aiida.orm import QueryBuilder, Code
import pandas as pd

qb = QueryBuilder()
qb.append(Code, project=["id", "label", "attributes.computer_id", "attributes.filepath_executable", "attributes.default_calc_job_plugin"])
rows = qb.all()

if rows:
    df = pd.DataFrame(rows, columns=["PK", "Label", "Computer ID", "Executable", "Default Plugin"])
    display(df)
else:
    print("No AiiDA codes configured yet. Click 'Set up Local FEFF Code' above to get started.")